In [1]:
import angr
import numpy as np
import os
binary_path = os.path.join('.', 'Dataset-1', 'openssl', 'x64-clang-3.5-O0_afalg.so')
# 加载二进制文件
project = angr.Project(binary_path, auto_load_libs=False)

# 构建控制流图(CFG)
cfg = project.analyses.CFGFast()

In [2]:
from scipy import sparse

func = cfg.kb.functions.function(name="EVP_CIPHER_CTX_iv_noconst")
block_addrs = [block.addr for block in func.blocks]
addr_to_idx = {addr: idx for idx, addr in enumerate(block_addrs)}
n_blocks = len(block_addrs)
entry_node = func.startpoint
# 初始化邻接矩阵
adj_matrix = sparse.lil_matrix((n_blocks, n_blocks), dtype=int)
# 填充邻接矩阵（稀疏矩阵版本）
for block in func.blocks:
    node = cfg.model.get_any_node(block.addr)
    succ = getattr(node, 'successors', [])
    for succ_node in succ:
        if succ_node.addr in block_addrs:
            src_idx = addr_to_idx[block.addr]
            dest_idx = addr_to_idx[succ_node.addr]
            adj_matrix[src_idx, dest_idx] = 1


In [6]:
for insn in block.capstone.insns:
    print(insn)

0x401620:	jmp	qword ptr [rip + 0x2039fa]


In [ ]:
from normalize_instr import *
# 示例用法
binary = "./Dataset-1/clamav/x86-gcc-9-O3_sigtool"  # 替换为你的二进制文件路径

# 通过函数名反汇编
disassemble_function(binary, function_name="main")

[+] 找到目标函数: main @ 0x41d000
    函数大小: 6832 字节
    基本块数量: 499

======= 汇编代码 (原始 => 归一化) =======

; 基本块 0x41d000 - 0x41d015
0x41d000: lea ecx, [esp + 4]             => lea ecx, [esp+<POSITIVE>]
0x41d004: and esp, 0xfffffff0            => and esp, <NEGATIVE>
0x41d007: push dword ptr [ecx - 4]       => push [ecx+<NEGATIVE>]
0x41d00a: push ebp                       => push ebp
0x41d00b: mov ebp, esp                   => mov ebp, esp
0x41d00d: push edi                       => push edi
0x41d00e: push esi                       => push esi
0x41d00f: push ebx                       => push ebx
0x41d010: call 0x41eaf0                  => call <NEAR>

; 基本块 0x41d015 - 0x41d037
0x41d015: add ebx, 0x2cdfeb              => add ebx, <POSITIVE>
0x41d01b: push ecx                       => push ecx
0x41d01c: sub esp, 0x358                 => sub esp, <POSITIVE>
0x41d022: mov esi, dword ptr [ecx]       => mov esi, [ecx]
0x41d024: mov edi, dword ptr [ecx + 4]   => mov edi, [ecx+<POSITIVE>]
0x41d027: mov ea